# 04 — Explore (find the story)

The **exploration** stage: try cuts of the income-disparity data and decide the story + which
charts tell it, *before* building anything publication-ready in `06-viz-social`. Nothing here is
final — it's for deciding framing.

## Rendering engine note

Charts use the **shared Pillow factory**, not matplotlib. This project's `.venv` is Python 3.14,
where matplotlib 3.10 hits a `RecursionError` on tick construction (a known trap, also seen in
dungeon-crawler-carl / video-game-scores). Exploring on the same Pillow templates `06-viz-social`
will use means *what we explore is what ships*. Charts display inline via `render_chart(..., filename=None)`.

DuckDB is opened **read-only** (viz reads, never writes) and closed in the Cleanup cell.

## ⚠️ Read first: the two Gini metrics are different measurements, not one number

Every Gini here is 0–100 (0 = perfect equality, 100 = one person holds everything), but the World
Bank's series is built from **two different kinds of household survey that measure two different
things.** This is the defining caveat of the whole project, so it lives here in the notebook, not
just in a chart subtitle.

**Income-based Gini** — how unequally **income** is distributed (wages, self-employment, investment
returns, pensions and cash transfers, usually *after* taxes and benefits). Collected via **household
income surveys** that ask what people *earned and received*. Used mostly by **high-income countries**
(Europe, North America, most of Latin America).

**Consumption-based Gini** — how unequally **spending/consumption** is distributed (food, housing,
goods, services, sometimes incl. the value of home-grown food). Collected via **household budget /
expenditure surveys** that record what people *bought and consumed*. Used mostly by **low- and
middle-income countries** (most of Africa, South Asia), where much income is informal, seasonal, or
in-kind and hard to measure — but spending is observable.

**Why they are NOT directly comparable.** Consumption is smoother than income: households save in
good years and draw down/borrow in bad years, the rich save a large share of income (so their
*consumption* looks closer to everyone else's than their *income* does), and the poor spend nearly
everything. So for the same population a **consumption Gini comes out lower than an income Gini** —
the World Bank's rule of thumb is **~4.7 points lower on average** (up to ~10 in some regions).
Ranking a consumption-survey country against an income-survey one therefore bakes in a gap that is
*purely about which survey they run*. Two biases compound it: income surveys tend to **miss the very
rich** (undercount top incomes → understate inequality), and equivalence-scale/definition choices
differ.

**Consequence for these charts (owner decision, this session):** we do **not** put the two metrics
in one ranking. We show **income-based and consumption-based views separately**, and where we place
them together it's an explicit **side-by-side** with the metric difference stated in the subtitle —
never a single mixed list.

**Bigger caveat to state publicly: neither series is a complete picture of global inequality.**
- Each is a *partial* dataset — only 68 countries have a comparable income Gini and 102 a consumption
  Gini (45 of 216 countries have neither in this pull). There is **no single harmonized measure that
  covers all countries the same way.**
- Both rely on **household surveys**, which miss the very top of the distribution (the ultra-wealthy
  are undersampled and under-report), so measured inequality is, if anything, **understated**.
- Ginis are measured only in **irregular survey years** (our "latest" can be several years old) and
  capture inequality *within* a country, saying nothing about inequality *between* countries.
- So any chart here is "the best comparable snapshot the World Bank publishes," **not** a definitive
  ranking of global inequality. That framing goes on the published charts and in the README.

**What the number means, on every chart.** A cold reader shouldn't have to know the dataset, so
each chart's subtitle leads with the plain-English Gini definition: *0–100, where 0 = everyone equal
and 100 = one person holds everything; higher = more unequal.* Keep that lead on the social/web
renders too — it's the single most important piece of context for someone arriving from a link.

In [ ]:
import sys, os
from pathlib import Path
import duckdb, pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))  # workspace shared/

from chart_factory import render_chart
from colors import c

con = duckdb.connect('data/project.duckdb', read_only=True)

# One color per metric, used consistently across every chart in this notebook.
INCOME_COLOR = c('gold')       # income-based
CONSUMPTION_COLOR = c('teal')  # consumption-based
SRC = 'World Bank WDI + PIP (survey years vary)  ·  @unwelcomedata'
print('connected (read-only); shared factory loaded')

## The exploration table

One row per country with its **latest** Gini, its year, its welfare metric, and latest values of the
other indicators (the `countries_latest` snapshot from `03-prepare`). We split it immediately into
the two metric groups and keep them separate from here on.

In [ ]:
snap = pd.read_parquet('data/processed/countries_latest.parquet')
g = snap.dropna(subset=['gini_index']).copy()

income = g[g.gini_welfare_type == 'income'].copy()
consumption = g[g.gini_welfare_type == 'consumption'].copy()
unknown = g[g.gini_welfare_type.isna()]

print(f'countries with a Gini: {len(g)} of {len(snap)}  '
      f'(income {len(income)}, consumption {len(consumption)}, metric unknown {len(unknown)})')
print(f'countries with NO Gini at all: {snap.gini_index.isna().sum()}')
print(f'\nmean latest Gini — income: {income.gini_index.mean():.1f}, '
      f'consumption: {consumption.gini_index.mean():.1f} '
      f'(income runs higher, as expected)')

## Chart 1 — Side-by-side: most unequal by each metric (NOT a merged ranking)

The comparison you asked for: the most unequal **consumption-based** countries beside the most
unequal **income-based** countries, on a **shared x-scale** so the panels are visually honest. The
subtitle states who typically uses which metric. This is a *comparison of two partial datasets*,
deliberately not combined into one list.

In [ ]:
n = 12
cons_top = consumption.nlargest(n, 'gini_index').copy()
inc_top = income.nlargest(n, 'gini_index').copy()
for d in (cons_top, inc_top):
    d['label'] = d['gini_index'].map(lambda v: f'{v:.1f}')

render_chart({
    'type': 'side_by_side_bars',
    'table_left': cons_top, 'table_right': inc_top,
    'category_col': 'country_name', 'value_col': 'gini_index', 'label_col': 'label',
    'left_title': 'Consumption-based', 'right_title': 'Income-based',
    'left_color': CONSUMPTION_COLOR, 'right_color': INCOME_COLOR,
    'title': 'Most unequal countries — by survey type, shown separately',
    'subtitle': ('Gini index 0–100: 0 = everyone equal, 100 = one person has it all (higher = more unequal). '
                 'Consumption surveys (lower-income) vs income surveys (richer) are not comparable — not merged.'),
    'source': SRC, 'filename': None,
})

## Chart 2 — Most unequal, **consumption-based only** (internally comparable)

A clean single-metric ranking: only consumption-survey countries, so every bar IS comparable to
every other. This is the developing-world inequality picture (Southern Africa dominates).

In [ ]:
cons15 = consumption.nlargest(15, 'gini_index').copy()
cons15['label'] = cons15['gini_index'].map(lambda v: f'{v:.1f}')
render_chart({
    'type': 'single_ranked_bars', 'table': cons15,
    'category_col': 'country_name', 'value_col': 'gini_index', 'label_col': 'label',
    'bar_color': CONSUMPTION_COLOR,
    'title': 'Most unequal countries — consumption-based Gini',
    'subtitle': ('Gini index 0–100: 0 = everyone equal, 100 = one person has it all (higher = more unequal). '
                 'Consumption surveys only (mostly lower/middle-income) — all bars comparable; 102 countries.'),
    'source': SRC, 'filename': None,
})

## Chart 3 — Most unequal, **income-based only** (internally comparable)

The same, for income-survey countries — the richer-world + Latin America picture. Latin America
(Colombia, Brazil, Panama, Costa Rica) leads among income-based measures.

In [ ]:
inc15 = income.nlargest(15, 'gini_index').copy()
inc15['label'] = inc15['gini_index'].map(lambda v: f'{v:.1f}')
render_chart({
    'type': 'single_ranked_bars', 'table': inc15,
    'category_col': 'country_name', 'value_col': 'gini_index', 'label_col': 'label',
    'bar_color': INCOME_COLOR,
    'title': 'Most unequal countries — income-based Gini',
    'subtitle': ('Gini index 0–100: 0 = everyone equal, 100 = one person has it all (higher = more unequal). '
                 'Income surveys only (richer countries + most of Latin America) — all bars comparable; 68 countries.'),
    'source': SRC, 'filename': None,
})

## Chart 4 — Side-by-side: most EQUAL by each metric

The equal end, again kept separate by metric. Watch India appear among the consumption-based most
equal (25.5) — a good teaching case: a consumption survey tends to *understate* inequality vs an
income survey, so India's low value is partly a metric effect, not purely a real-world one.

In [ ]:
cons_eq = consumption.nsmallest(12, 'gini_index').sort_values('gini_index', ascending=False).copy()
inc_eq = income.nsmallest(12, 'gini_index').sort_values('gini_index', ascending=False).copy()
for d in (cons_eq, inc_eq):
    d['label'] = d['gini_index'].map(lambda v: f'{v:.1f}')
render_chart({
    'type': 'side_by_side_bars',
    'table_left': cons_eq, 'table_right': inc_eq,
    'category_col': 'country_name', 'value_col': 'gini_index', 'label_col': 'label',
    'left_title': 'Consumption-based', 'right_title': 'Income-based',
    'left_color': CONSUMPTION_COLOR, 'right_color': INCOME_COLOR,
    'title': 'Most equal countries — by survey type, shown separately',
    'subtitle': ('Gini index 0–100: lower = more equal (0 = everyone equal, 100 = one person has it all). '
                 'India’s low consumption-based value partly reflects the metric, not only real equality.'),
    'source': SRC, 'filename': None,
})

## Chart 5 — Average Gini by region, split by metric

Regional pattern, but WITHOUT mixing metrics into one bar (the earlier version's flaw). Each region
shows its consumption-based mean beside its income-based mean, so you compare like with like. Some
region×metric cells are thin (few countries) — exploratory only.

In [ ]:
def region_means(df):
    r = (df.groupby('region', as_index=False)['gini_index'].mean())
    r['label'] = r['gini_index'].map(lambda v: f'{v:.1f}')
    return r

reg_c = region_means(consumption).sort_values('gini_index', ascending=False)
reg_i = region_means(income).sort_values('gini_index', ascending=False)
print('consumption by region:\n', reg_c[['region','gini_index']].to_string(index=False))
print('\nincome by region:\n', reg_i[['region','gini_index']].to_string(index=False))
render_chart({
    'type': 'side_by_side_bars',
    'table_left': reg_c, 'table_right': reg_i,
    'category_col': 'region', 'value_col': 'gini_index', 'label_col': 'label',
    'left_title': 'Consumption-based', 'right_title': 'Income-based',
    'left_color': CONSUMPTION_COLOR, 'right_color': INCOME_COLOR,
    'title': 'Average inequality by region — metrics kept separate',
    'subtitle': ('Gini index 0–100: higher = more unequal. Mean latest Gini per region, split by survey type. '
                 'Regions with few countries of a given metric are thin — indicative, not definitive.'),
    'source': SRC, 'filename': None,
})

## Chart 6 — Does wealth buy equality? Gini vs GDP per capita (income-based)

Because we must not pool metrics, this relationship is explored **within each metric separately.**
Among income-survey countries the link is clear: **richer → more equal (r ≈ −0.51).**

In [ ]:
inc_gdp = income[income.gdp_per_capita_ppp.notna()].copy()
print('income-based, n =', len(inc_gdp), '| r =', round(inc_gdp.gini_index.corr(inc_gdp.gdp_per_capita_ppp), 3))
render_chart({
    'type': 'scatter_plot', 'table': inc_gdp,
    'x_col': 'gdp_per_capita_ppp', 'y_col': 'gini_index', 'label_col': 'country_name',
    'label_loners': 8,
    'x_axis_label': 'GDP per capita, PPP (current int$)', 'y_axis_label': 'Gini index (0–100)',
    'point_color': INCOME_COLOR, 'x_fmt': (lambda v: f'${v/1000:,.0f}k'),
    'title': 'Wealth vs inequality — income-based countries',
    'subtitle': ('Gini 0–100 (higher = more unequal). Richer income-survey countries are clearly more '
                 'equal (r ≈ −0.51). Income metric only.'),
    'source': SRC, 'filename': None,
})

…and among **consumption-based** countries the link is much weaker (r ≈ −0.23) — a real finding,
and exactly why pooling the two metrics into one scatter would have been misleading.

In [ ]:
cons_gdp = consumption[consumption.gdp_per_capita_ppp.notna()].copy()
print('consumption-based, n =', len(cons_gdp), '| r =', round(cons_gdp.gini_index.corr(cons_gdp.gdp_per_capita_ppp), 3))
render_chart({
    'type': 'scatter_plot', 'table': cons_gdp,
    'x_col': 'gdp_per_capita_ppp', 'y_col': 'gini_index', 'label_col': 'country_name',
    'label_loners': 8,
    'x_axis_label': 'GDP per capita, PPP (current int$)', 'y_axis_label': 'Gini index (0–100)',
    'point_color': CONSUMPTION_COLOR, 'x_fmt': (lambda v: f'${v/1000:,.0f}k'),
    'title': 'Wealth vs inequality — consumption-based countries',
    'subtitle': ('Gini 0–100 (higher = more unequal). Weaker link among consumption-survey countries '
                 '(r ≈ −0.23). Consumption metric only.'),
    'source': SRC, 'filename': None,
})

## What the exploration suggests (for owner review)

The through-line is honest: **we compare like-with-like and never merge the two metrics.** Candidate
stories, strongest first:

1. **The metric split itself is the smart story** — "there is no single global inequality number."
   The side-by-side (Chart 1) + the two GDP scatters (Charts 6–7, r ≈ −0.51 income vs −0.23
   consumption) make the point that *how you measure changes what you see*, and that both datasets
   are partial. Most on-brand.
2. **Two clean single-metric rankings** (Charts 2–3): the developing-world (consumption) and
   richer-world + Latin America (income) inequality leaders, each internally comparable.
3. **Regional split** (Chart 5) as supporting context.

Open framing questions:
- Lead with the side-by-side comparison, or with the "how you measure changes the answer" scatter pair?
- Publish both single-metric rankings (Charts 2 & 3) as a pair, or just the side-by-side (Chart 1)?
- Keep the regional split (Chart 5), or drop it as too thin per region×metric?
- Confirm the incompleteness framing (partial coverage, top-income undercount, survey-year lag)
  should headline the README, not just footnote it.

**Next step is NOT to build social charts.** Per the workflow, pause here for owner review of the
framing; only after that build the chosen charts in `06-viz-social`.

---

## Browse: every country's trajectory over time (small multiples)

These are **browsing** charts, not candidate social charts — a way to eyeball how each indicator has
moved for *every* country at once. A single overlaid line chart with ~200 countries would be an
unreadable spaghetti plot, so instead we draw a **small-multiples grid**: one tiny line panel per
country (alphabetical), on a **shared y-axis per chart** so trajectories are visually comparable
across countries. The images are deliberately **very tall** — scroll to browse.

The shared chart factory has no faceting/small-multiples template, and this is exploration-only, so
the grid is drawn by a small **notebook-local** Pillow helper below (matplotlib is unusable on this
project's Python 3.14 venv). If any of these graduate toward a social story, we'd build a proper
reusable template in `shared/` at that point — for now it stays local to exploration.

In [ ]:
# Browse/standout/dashboard charts use the SHARED factory templates
# (small_multiples_grid, annotated_small_multiples, metric_dashboard) — promoted
# from earlier notebook-local helpers so 06-viz-social + the per-country build reuse them.
panel = con.execute('SELECT iso_alpha3, country_name, year, fertility_rate, '
                    'life_expectancy, unemployment_pct FROM countries_clean').df()
# wider panel (adds GDP + Gini + urban) for the US dashboard
panel_full = con.execute('SELECT iso_alpha3, country_name, year, fertility_rate, '
                         'life_expectancy, unemployment_pct, gdp_per_capita_ppp, gini_index, '
                         'urban_pct FROM countries_clean').df()
print('panel rows:', len(panel))

### Browse 1 — Fertility rate over time, by country

Births per woman, 1960–2024, one panel per country (shared y-axis 0.6–8.9). The near-universal
downward slope — the global fertility decline — is the thing to eyeball here. Tall image: scroll.

In [ ]:
render_chart({
    'type': 'small_multiples', 'table': panel,
    'entity_col': 'country_name', 'x_col': 'year', 'value_col': 'fertility_rate',
    'unit': 'births/woman', 'line_color': c('navy'),
    'title': 'Fertility rate over time — births per woman, by country',
    'subtitle': 'One panel per country, shared y-axis. Corner number = latest value.',
    'source': SRC, 'filename': None,
})

### Browse 2 — Life expectancy over time, by country

Years at birth, 1960–2024, shared y-axis 11–86.5. Watch for the broad upward march plus the visible
shocks (dips) some countries take — wars, famines, epidemics.

In [ ]:
render_chart({
    'type': 'small_multiples', 'table': panel,
    'entity_col': 'country_name', 'x_col': 'year', 'value_col': 'life_expectancy',
    'unit': 'years', 'line_color': c('navy'),
    'title': 'Life expectancy over time — years at birth, by country',
    'subtitle': 'One panel per country, shared y-axis. Corner number = latest value.',
    'source': SRC, 'filename': None,
})

### Browse 3 — Unemployment rate over time, by country

% of labor force (ILO modeled estimate), 1991–2025 (shorter series than the others), shared y-axis
0.1–38.8. Note this is an ILO *model*, not raw national figures, and fewer countries have it (186).

In [ ]:
render_chart({
    'type': 'small_multiples', 'table': panel,
    'entity_col': 'country_name', 'x_col': 'year', 'value_col': 'unemployment_pct',
    'unit': '%', 'line_color': c('navy'),
    'title': 'Unemployment rate over time — % of labor force (ILO estimate), by country',
    'subtitle': 'One panel per country, shared y-axis. Corner number = latest value.',
    'source': SRC, 'filename': None,
})

---

## Standout: life-expectancy shocks (a candidate social angle)

The small-multiples browse surfaced the most dramatic stories in the data: countries whose
life-expectancy line **falls off a cliff and recovers** — each a real historical catastrophe.
These are **independent examples of the same phenomenon**, so they belong in **small multiples**
(one panel each, its own y-scale), NOT overlaid on a shared axis — and each dip is annotated with
the event that caused it. No reference country is mixed in. This is exploration of a possible social
framing, not a final social chart.

The clean examples: **Cambodia** (Khmer Rouge, 1975–79), **Rwanda** (1994 genocide), and
**Timor-Leste** (Indonesian invasion/occupation from 1975) — each an unambiguous cliff to a
catastrophic low that then recovers. Ukraine is left out — its post-Soviet decline is a slow sag,
not a cliff, so it doesn't fit the "catastrophe" framing.

> **Verified against source (this session).** The earlier draft included **Central African Republic**
> (a 2009 ~15-yr low, marked ⚠ verify). Tie-back to the raw World Bank series confirmed CAR's
> life-expectancy series is a **data artifact** — it whipsaws to demographically impossible values
> (2009 = 14.7, 2019 = 31.5, 2022 = 18.8), a modeling breakdown for a conflict state with weak vital
> registration, NOT a real event. **CAR is dropped** (and its dashboard life-exp panel is suppressed).
> All 14,006 life-expectancy country-years otherwise tie back to source exactly (zero mismatches).

**Two candidate framings to compare below (owner to pick before social):**

1. **Clean 3-panel** — Cambodia / Rwanda / Timor-Leste. Every panel is a dramatic cliff; tightest,
   most consistent story.
2. **4-panel with Syria** — adds **Syria** (civil war; low point 2015 = 63.3). Syria's series is
   clean and ties to source (rise to ~73.5 by 2010, war-driven drop, recovery), and it's recent and
   recognizable — but its dip is **shallower** (a ~10-yr dent, not a single-digit collapse), so it's
   a gentler peer next to the other three. Each panel is on its own y-scale, so it still reads.

In [ ]:
# VARIANT 1 — clean 3-panel: unambiguous cliffs only. Each panel its own y-scale,
# each dip annotated with its event. CAR dropped (confirmed data artifact, see above).
catastrophe_specs_3 = [
    {'entity': 'Cambodia',    'name': 'Cambodia',    'event': 'Khmer Rouge',         'mark_x': 1977},
    {'entity': 'Rwanda',      'name': 'Rwanda',      'event': 'Genocide',            'mark_x': 1994},
    {'entity': 'Timor-Leste', 'name': 'Timor-Leste', 'event': 'Invasion/occupation', 'mark_x': 1978},
]
render_chart({
    'type': 'annotated_small_multiples', 'table': panel,
    'entity_col': 'country_name', 'x_col': 'year', 'value_col': 'life_expectancy',
    'specs': catastrophe_specs_3, 'line_color': c('navy'),
    'title': 'When catastrophe collapses life expectancy',
    'subtitle': 'Life expectancy at birth (years). Each panel is one country on its own scale; the marked year is the low point.',
    'source': SRC, 'filename': None,
})

In [ ]:
# VARIANT 2 — 4-panel adding Syria (civil war; low point 2015 = 63.3). Syria's series is
# clean and ties to source; its dip is shallower than the other three but reads on its own scale.
catastrophe_specs_4 = [
    {'entity': 'Cambodia',            'name': 'Cambodia',    'event': 'Khmer Rouge',         'mark_x': 1977},
    {'entity': 'Rwanda',              'name': 'Rwanda',      'event': 'Genocide',            'mark_x': 1994},
    {'entity': 'Timor-Leste',         'name': 'Timor-Leste', 'event': 'Invasion/occupation', 'mark_x': 1978},
    {'entity': 'Syrian Arab Republic', 'name': 'Syria',      'event': 'Civil war',           'mark_x': 2015},
]
render_chart({
    'type': 'annotated_small_multiples', 'table': panel,
    'entity_col': 'country_name', 'x_col': 'year', 'value_col': 'life_expectancy',
    'specs': catastrophe_specs_4, 'line_color': c('navy'),
    'title': 'When catastrophe collapses life expectancy',
    'subtitle': 'Life expectancy at birth (years). Each panel is one country on its own scale; the marked year is the low point.',
    'source': SRC, 'filename': None,
})

---

## US-only dashboard: the United States across the key metrics

A **US-only** view — the US trajectory on each key metric, shown together as a dashboard. No other
country's line appears on these panels (the cross-country comparisons live in the charts above).
Two pieces:

- **Dashboard** — one small line panel per metric (life expectancy, GDP per capita PPP, Gini,
  fertility, unemployment, urban population %), each the US series over time. Axes are **0-based**
  (not zoomed to the data range) so spikes and dips read at their true scale, not exaggerated. The
  COVID dip is annotated on life expectancy.
- **Where the US ranks** — a single US-only summary bar chart of the US's **global percentile** on
  each metric (0th = lowest in the world, 100th = highest). Still US-only: it's the US's rank, not
  other countries drawn alongside. **Gini is ranked against income-based countries only** (the US is
  income-based; ranking it against consumption-survey countries would mix non-comparable metrics).

The profile is the story: the US is **extremely rich (95th pct) and unusually unequal for a rich
country (~79th pct on Gini, vs income-based peers), but only middling on life expectancy (74th pct)**
— rich, unequal, and not as healthy as its wealth would predict. Latest US values: GDP ~$90k PPP,
Gini 41.8 (income-based), life expectancy 78.9 yrs, fertility 1.63, unemployment 4.2%.

**Annotations are fact-based only.** Clear, established events are marked (COVID on life expectancy;
the 2008 financial crisis and COVID on unemployment). Source-documented **data breaks** are marked
too — the US Gini series has a World Bank comparability break at **2002** (survey redesign; pre/post
not directly comparable), shown as a dashed line. We do NOT annotate speculative causes: e.g. the
1960s–70s fertility decline is left unlabeled because no single clear event explains it (baby-boom
unwinding, women entering the workforce, and contraception access all overlap).

In [ ]:
# US-only dashboard via the shared metric_dashboard template (the reusable entity-profile chart).
# Annotations are FACT-BASED ONLY: clear established events (COVID, 2008 crisis) as point markers,
# and source-documented series breaks (US Gini survey redesign 2002) as dashed lines. NO speculative
# causes — e.g. fertility's 1960s decline gets no cause label (no single clear event).
us_metrics = [
    {'col': 'life_expectancy',    'name': 'Life expectancy',   'unit': 'years',        'annot': [(2021, 'COVID')]},
    {'col': 'gdp_per_capita_ppp', 'name': 'GDP per capita',    'unit': 'PPP int$'},
    {'col': 'gini_index',         'name': 'Income inequality', 'unit': 'Gini (income-based)',
     'breaks': [(2002, 'survey redesign — pre/post not comparable')]},
    {'col': 'fertility_rate',     'name': 'Fertility rate',    'unit': 'births/woman'},
    {'col': 'unemployment_pct',   'name': 'Unemployment',      'unit': '% labor force',
     'annot': [(2010, '9.6% — post-2008 crisis'), (2020, '8.1% — COVID')]},
    {'col': 'urban_pct',          'name': 'Urban population',   'unit': '% of total'},
]
us_only = panel_full[panel_full.iso_alpha3 == 'USA'].copy()
render_chart({
    'type': 'metric_dashboard', 'table': us_only, 'metrics': us_metrics,
    'line_color': c('navy'),
    'title': 'United States — the key metrics over time',
    'subtitle': 'US only. Each panel is 0-based (true scale, not zoomed). Gini is income-based.',
    'source': SRC, 'filename': None,
})

In [ ]:
# US-only summary: the US's GLOBAL PERCENTILE on each metric (US's rank, not other countries drawn).
import pandas as pd
snap_l = pd.read_parquet('data/processed/countries_latest.parquet')

def us_pctile(col, pool=None):
    # pool = optional filtered DataFrame to rank against (else all countries)
    src = pool if pool is not None else snap_l
    s = src[col].dropna()
    usv = snap_l.loc[snap_l.iso_alpha2=='US', col]
    if usv.empty or pd.isna(usv.iloc[0]): return None, None
    usv = usv.iloc[0]
    return round(100.0 * (s <= usv).mean(), 0), usv

# US Gini is income-based, so rank it ONLY against other income-based countries — never
# combine income + consumption surveys (they're not comparable).
income_pool = snap_l[snap_l['gini_welfare_type'] == 'income']
rows = []
for col, lbl, pool in [('gdp_per_capita_ppp','GDP per capita (PPP)', None),
                       ('gini_index','Income inequality (Gini, income-based)', income_pool),
                       ('life_expectancy','Life expectancy', None),
                       ('fertility_rate','Fertility rate', None),
                       ('unemployment_pct','Unemployment', None)]:
    p, usv = us_pctile(col, pool)
    rows.append({'indicator': lbl, 'pct': p, 'label': f'{p:.0f}th pct', 'us_value': usv})
us = pd.DataFrame(rows).sort_values('pct', ascending=False)
print(us[['indicator','us_value','pct']].to_string(index=False))

render_chart({
    'type': 'single_ranked_bars', 'table': us,
    'category_col': 'indicator', 'value_col': 'pct', 'label_col': 'label',
    'bar_color': c('navy'),
    'title': 'Where the United States ranks in the world',
    'subtitle': ('US global percentile per metric (100th = highest, 0th = lowest). Gini ranked vs '
                 'income-based countries only. Rich (95th) and unequal (~79th) but middling on life expectancy (74th).'),
    'source': SRC, 'filename': None,
})

## Cleanup

In [ ]:
con.close()
print('Connection closed.')